## Dataset Testing Notebooks

This notebook is for testing and understanding the setup and organization of the sequence data from MPC sim.

This set will also be used to take the sequence data, clean it, and convert it to tensors or another format for use with pytorch training

In [1]:
#imports 
import numpy as np
import os 

## Section 1: Loading a single datset and inspecting data 

In [24]:
#set home 
from pathlib import Path

# Get the project root (parent of notebooks directory)
project_root = Path.cwd().parent
print(f"Project root: {project_root}")

#set paths for single map, spielberg
single_map_path = "/home/devin_work/work/f1tenth/ApproxiMPC/mpc_datasets/run_20260420T215212Z/Nuerburgring/reverse/stmpc_Nuerburgring_reverse.npz"
print(f"Single map path: {single_map_path}")

#unzip .npz option
data = np.load(single_map_path)

Project root: /home/devin_work/work/f1tenth/ApproxiMPC/LSTM_training
Single map path: /home/devin_work/work/f1tenth/ApproxiMPC/mpc_datasets/run_20260420T215212Z/Nuerburgring/reverse/stmpc_Nuerburgring_reverse.npz


In [3]:
#view and organize different arrays 
print(data.files)

for key in data.files:
    print(f"Key: {key}, Shape: {data[key].shape}, Dtype: {data[key].dtype}")


['feature_names', 'is_perturbed', 'expert_actions', 'noise_vectors', 'path_curvature_lookahead', 'episode_ids', 'terminations', 'step_ids', 'lidar_scans', 'next_observations', 'action_names', 'lidar_names', 'collision_flags', 'path_curvature_lookahead_names', 'collector_lap_counts', 'path_kappa_current', 'path_centerline_idx_anchor', 'next_lidar_scans', 'path_curvature_lookahead_m', 'executed_actions', 'path_s_anchor', 'env_lap_counts', 'rewards', 'boundary_flags', 'observations', 'stmpc_status_codes', 'truncations']
Key: feature_names, Shape: (5,), Dtype: <U12
Key: is_perturbed, Shape: (197175,), Dtype: bool
Key: expert_actions, Shape: (197175, 2), Dtype: float32
Key: noise_vectors, Shape: (197175, 2), Dtype: float32
Key: path_curvature_lookahead, Shape: (197175, 5), Dtype: float32
Key: episode_ids, Shape: (197175,), Dtype: int32
Key: terminations, Shape: (197175,), Dtype: bool
Key: step_ids, Shape: (197175,), Dtype: int32
Key: lidar_scans, Shape: (197175, 60), Dtype: float16
Key: nex

In [4]:
#inspect individual data types
print("observations names")
print(data["feature_names"])
print("action names")
print(data["action_names"])
# print(data["expert_actions"])
# print(data["executed_actions"])
print("lidar")
print(data["lidar_names"])
#only 60 lidar scans? double check
# print(data["lidar_scans"])


observations names
['pose_x' 'pose_y' 'delta' 'linear_vel_x' 'pose_theta']
action names
['steering_angle' 'speed']
lidar
['lidar_0' 'lidar_1' 'lidar_2' 'lidar_3' 'lidar_4' 'lidar_5' 'lidar_6'
 'lidar_7' 'lidar_8' 'lidar_9' 'lidar_10' 'lidar_11' 'lidar_12' 'lidar_13'
 'lidar_14' 'lidar_15' 'lidar_16' 'lidar_17' 'lidar_18' 'lidar_19'
 'lidar_20' 'lidar_21' 'lidar_22' 'lidar_23' 'lidar_24' 'lidar_25'
 'lidar_26' 'lidar_27' 'lidar_28' 'lidar_29' 'lidar_30' 'lidar_31'
 'lidar_32' 'lidar_33' 'lidar_34' 'lidar_35' 'lidar_36' 'lidar_37'
 'lidar_38' 'lidar_39' 'lidar_40' 'lidar_41' 'lidar_42' 'lidar_43'
 'lidar_44' 'lidar_45' 'lidar_46' 'lidar_47' 'lidar_48' 'lidar_49'
 'lidar_50' 'lidar_51' 'lidar_52' 'lidar_53' 'lidar_54' 'lidar_55'
 'lidar_56' 'lidar_57' 'lidar_58' 'lidar_59']


In [5]:
#new stuff from dataset 2
print("cureve lookaheada")
print(data["path_curvature_lookahead"])

print("path kappa")
print(data["path_kappa_current"])

print("centerline idx anchor")
print(data["path_centerline_idx_anchor"])

print("path curve lookahead m")
print(data["path_curvature_lookahead_m"])

print("stmpc status")
print(data["stmpc_status_codes"])

print("anchor ")
print(data["path_s_anchor"])

cureve lookaheada
[[-0.01118145 -0.00751788 -0.00379293  0.0035727  -0.01106888]
 [-0.01083469 -0.00714665 -0.00342044  0.00400111 -0.01187354]
 [-0.01083469 -0.00714665 -0.00342044  0.00400111 -0.01187354]
 ...
 [-0.01118145 -0.00751788 -0.00379293  0.0035727  -0.01106888]
 [-0.01118145 -0.00751788 -0.00379293  0.0035727  -0.01106888]
 [-0.01118145 -0.00751788 -0.00379293  0.0035727  -0.01106888]]
path kappa
[-0.01342908 -0.01347669 -0.01347669 ... -0.01342908 -0.01342908
 -0.01342908]
centerline idx anchor
[0 1 1 ... 0 0 0]
path curve lookahead m
[1. 2. 3. 5. 8.]
stmpc status
[0 0 0 ... 0 0 0]
anchor 
[0.0000000e+00 2.9951580e-02 5.9842102e-02 ... 3.5622971e+02 3.5625323e+02
 3.5627682e+02]


In [25]:
import numpy as np

# Select all rows (:) and the second column (index 1)
print(np.max(data["expert_actions"][:, 1]))

3.0211124


## What columns and data to keep

### Inputs
1. observations: pose (x,y,theta), yaw, and linear velocity. Drop x and y coordinates. Keep theta (current steering angle), yaw angle, and current linear velocity
3. lidar_scans: current timestep scans, check with henry about number of rays 
3. episode_id: useful for splitting up train, val, and test
8. step_ids: not used as input, used to organize overall rows (for human readability)

### Additional Keep, potential inputs?

1. Curve lookahead, says the curvature of the next 5 or so paths
2. Centerline idx anchor, need to check, could be very useful. Could be extrapolated and used in actual inference alongside centerline distance 

### Outputs
1. expert_actions: desired output for the current timestep. speed and steering angle

### Drop, make note of them

9. feature_names: good to keep, helpful for reading data, never passed as inputs
10. action_names: good to keep, helpful for reading data, never passed as inputs
11. lidar_names: probably not necessary. But could be good to have 

### Drop
1. next observations: the next timestep of observations, unnecessary
2. executed_actions: desired action of the agent, before the execution DOUBLE CHECK (includes random noise)
3. rewards: rewards for MPC not useful for LSTM
4. is_perturbed: bool for MPC input, not needed for LSTM
5. next_lidar_scans: next timstep of lidar, not available during inferenc

Additional flags for MPC, should be unnecessary
1. terminations: check
2. truncations: check
3. collector_lap_counts: cehck
4. env_lap_counts: check
5. collision_flag: check
6. boundary_flags: check
7. noise_vectors: check




## Section 2: Data Cleaning and Prep Test

Drop the non needed columns

Save existing arrays, and make edits as needed (drop x and y)

Concatonate the arrays to a final object

Convert to a tensor

In [6]:
#info cols (for splits)
map_name = np.full(len(data["step_ids"]), "Spielberg_normal")
episodes = data["episode_ids"]
steps = data["step_ids"]

#input cols
observations = data["observations"]
lidar_scans = data["lidar_scans"]

#output col
expert_actions = data["expert_actions"]


In [7]:
#clean up observations 
observations = observations[:, 2:]
print(observations)

[[ 0.          3.          0.42493388]
 [ 0.          2.9762533   0.424936  ]
 [ 0.          2.9723322   0.4249397 ]
 ...
 [-0.35733333  2.5074775   0.8695752 ]
 [-0.32533333  2.4968202   0.85495454]
 [-0.35733333  2.485577    0.83904034]]


In [8]:
#concatenate input features (observations + lidar) along axis=1
# observations shape: (N, 3) -> [delta, linear_vel_x, pose_theta]
# lidar_scans shape: (N, 60) -> lidar measurements
# inputs = np.concatenate([observations, lidar_scans], axis=1)
# print(f"inputs shape: {inputs.shape}")  # Should be (N, 63)
# print(f"expert_actions shape: {expert_actions.shape}")  # Should be (N, 2)

# # Metadata stays separate for train/val/test splitting
# print(f"\nMetadata:")
# print(f"map_name: {map_name.shape}")
# print(f"episodes: {episodes.shape}")
# print(f"steps: {steps.shape}")

In [9]:
#convery to pytorch tensor!
#single tensor for each episode

#then assemble the tensor into a tensor of tensors 

## Section 3: Model Loading Test

In [10]:
# Section 3 execution: load saved model bundle and run a smoke-test inference
import sys
import importlib
from pathlib import Path
import torch

project_root = Path.cwd().parent
scripts_dir = project_root / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

import load_model as load_model_module
importlib.reload(load_model_module)
load_model_bundle_from_config = load_model_module.load_model_bundle_from_config

model_name = "LSTM_TEST"
run_date = sorted([d.name for d in (project_root / "models" / model_name).iterdir() if d.is_dir()])[-1]
config_path = project_root / "models" / model_name / run_date / f"{model_name}_config.json"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model, loaded_input_scaler, loaded_target_scaler, loaded_cfg = load_model_bundle_from_config(
    config_path=str(config_path),
    device=device,
)

model_cfg = loaded_cfg["model_config"]
input_dim = model_cfg["input_dim"]
dummy_x = torch.zeros((1, 100, input_dim), dtype=torch.float32, device=device)

with torch.no_grad():
    y_pred = loaded_model(dummy_x).cpu().numpy()

y_pred_phys = loaded_target_scaler.inverse_transform(y_pred)
print(f"Loaded model from: {config_path}")
print(f"Pred normalized: {y_pred}")
print(f"Pred physical units: {y_pred_phys}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/devin_work/work/f1tenth/ApproxiMPC/LSTM_training/models/LSTM_TEST'